# MYELIN-SR — Round 2 Training (Full Innovation Stack)

## What's new vs Round 1
| | Round 1 | Round 2 |
|---|---|---|
| Pixel loss | L1 | **Charbonnier** (smooth, better ternary gradients) |
| Frequency supervision | None | **FFT Frequency Loss** (directly targets sharpness) |
| Stiffness consolidation | rate=0.08, every 50 | **rate=0.02, every 100** |
| LR scheduler | CosineAnnealing (conflict) | **ReduceLROnPlateau** (cooperative) |
| Neural growth | None | **Dynamic GrowthModule** (frozen neurons spawn fresh children) |
| Patch training | Fixed 256px | **Progressive: 64→128→256** (curriculum) |
| Training data | DIV2K 800 imgs | **DIV2K + Flickr2K ~3,450 imgs** |

**Target:** PSNR ≥ 30.3 dB (Phase Gate 1.2) → ≥ 33.0 dB (Phase Gate 1.4)

In [ ]:
# ── Cell 1: Setup & GPU Check ──────────────────────────────────────────
import subprocess, sys
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU:     {props.name}')
    print(f'VRAM:    {props.total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Paths (FIXED) ──────────────────────────────────────────────
import os, sys
from pathlib import Path

# Kaggle base input
K_INPUT = Path('/kaggle/input')

# 1. Locate Codebase Dynamically
# This searches for 'myelin-sr-codebase' anywhere in /kaggle/input
code_search = list(K_INPUT.glob('**/myelin-sr-codebase'))
if not code_search:
    raise FileNotFoundError("Could not find 'myelin-sr-codebase' in /kaggle/input. "
                            "Ensure the dataset is added to the notebook.")
CODE_ROOT = code_search[0]

# 2. Update sys.path (FIX: Add ROOT and subfolders)
# Adding CODE_ROOT itself allows: from model import ...
sys.path.insert(0, str(CODE_ROOT)) 
sys.path.insert(0, str(CODE_ROOT / 'train'))
sys.path.insert(0, str(CODE_ROOT / 'eval'))

# 3. Define Work Directories
DATA_DIR = Path('/kaggle/working/data')
CKPT_DIR = Path('/kaggle/working/checkpoints')
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# 4. Locate Flickr2K Dynamically
flickr_search = list(K_INPUT.glob('**/flickr2k'))
FLICKR_INPUT_PATH = flickr_search[0] if flickr_search else None

print(f'Code Root: {CODE_ROOT}')
print(f'Flickr2K found at: {FLICKR_INPUT_PATH}')
print(f'Set14 expected at: {CODE_ROOT / "Set14"}')
# 3. Handle the Checkpoint (checkpoint-dlss)
RESUME_CKPT = None
# Search specifically in /kaggle/input for the checkpoint file
ckpt_search = list(K_INPUT.glob('**/checkpoint-dlss/myelin_sr_best.pt'))
if ckpt_search:
    RESUME_CKPT = None # FORCED CLEAN RUN
    # RESUME_CKPT = str(ckpt_search[0])
    print(f'Checkpoint found: {RESUME_CKPT}')
else:
    print('Checkpoint: not found, training from scratch')

In [ ]:
# ── Cell 3: Data Linking & Downloading (FIXED) ──────────────────────
import os
import urllib.request
import zipfile
import tarfile
from pathlib import Path

def download_and_extract(url, dest_dir, expected_subdir, expected_min=10):
    dest = Path(dest_dir)
    target = dest / expected_subdir
    existing = list(target.glob('**/*.png')) + list(target.glob('**/*.jpg'))
    if len(existing) >= expected_min:
        print(f'[OK] {expected_subdir}: {len(existing)} images found.')
        return

    filename = url.split('/')[-1]
    archive_path = dest / filename
    print(f'Downloading {filename}...')
    try:
        urllib.request.urlretrieve(url, archive_path)
        print(f'Extracting {filename}...')
        if archive_path.suffix == '.zip':
            with zipfile.ZipFile(archive_path, 'r') as z:
                z.extractall(dest)
        elif '.tar' in filename:
            with tarfile.open(archive_path, 'r') as t:
                t.extractall(path=dest)
        archive_path.unlink() 
        print(f'[OK] {expected_subdir} extracted.')
    except Exception as e:
        print(f'[ERROR] Failed to download {expected_subdir}: {e}')

def link_existing_data(source_path, dest_dir, folder_name):
    target = Path(dest_dir) / folder_name
    if target.exists() or target.is_symlink():
        print(f'[OK] {folder_name} already linked.')
        return 

    if source_path and Path(source_path).exists():
        print(f'Linking {folder_name} from {source_path}...')
        os.symlink(source_path, target)
        print(f'[OK] {folder_name} linked.')
    else:
        print(f'[SKIP] Source for {folder_name} not found at {source_path}')

# --- Execution ---

# 1. DIV2K: Download
download_and_extract(
    'http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip',
    DATA_DIR, 'DIV2K_train_HR', expected_min=700
)

# 2. Flickr2K: Link (Updated path based on your logs)
# Using the path found in your Cell 2: /kaggle/input/datasets/daehoyang/flickr2k
flickr_img_folder = Path('/kaggle/input/datasets/daehoyang/flickr2k/Flickr2K')
link_existing_data(flickr_img_folder, DATA_DIR, 'Flickr2K_HR')

# 3. Set14: Link from codebase
set14_source = CODE_ROOT / 'Set14'
link_existing_data(set14_source, DATA_DIR, 'Set14')

In [ ]:
import torch
import torch.nn as nn
from model import build_myelin_sr
from fpsan_conv2d import FPSANConv2d, NetworkGrowthManager
from losses import MyelinSRLoss
from metrics import calculate_psnr, calculate_ssim, PhaseGateValidator
from dataset import build_train_loader, build_val_loader

# Detect hardware
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_count = torch.cuda.device_count()
print(f'Device: {device} | GPUs detected: {gpu_count}')

# 1. Build the RAW model first for gate checks
raw_model = build_myelin_sr(2, 'quality').to(device)

# 2. Gate 1.1: Always validate against the raw_model
# DataParallel can mask attributes, leading to False failures
v = PhaseGateValidator()
assert v.check_valid_output(raw_model, device), 'Gate 1.1 FAILED'
print('Gate 1.1: PASS')

# 3. Verify GrowthModule availability
# This uses a dummy layer to confirm the logic independently of the main model
dummy_layer = FPSANConv2d(16, 32, 3, padding=1)
dummy_layer.stiffness.fill_(dummy_layer.stiffness_cap)
assert dummy_layer.is_saturated(), 'Saturation detection broken'
spawned = dummy_layer.spawn_growth_child()
assert spawned and dummy_layer.growth_child is not None, 'Growth spawn failed'
print(f'GrowthModule: PASS (child params={sum(p.numel() for p in dummy_layer.growth_child.parameters())})')

# 4. Activate Multi-GPU wrapping
# We keep raw_model as a reference for stats and wrap 'model' for training
if gpu_count > 1:
    print(f"Activating Multi-GPU training on {gpu_count} devices.")
    model = nn.DataParallel(raw_model)
else:
    model = raw_model

# Clean up temporary check objects
del dummy_layer

In [ ]:
# ── Cell 5: Training Config ────────────────────────────────────────────
CONFIG = {
    'scale':                  2,
    'quality':                'quality',
    'epochs':                 300,
    'batch_size':             16,
    'base_lr':                1e-4,

    # FIX 1: Slow consolidation
    'consolidation_rate':     0.02,
    'consolidation_interval': 100,

    # NEW: Dynamic growth
    'growth_threshold':       0.85,
    'growth_check_interval':  10,

    # NEW: Progressive patch size (curriculum learning)
    # epoch -> patch_size: 1-50: 64px, 51-150: 128px, 151+: 256px
    'patch_schedule':         [(1, 64), (51, 128), (151, 256)],

    # UPDATED losses
    'perceptual_weight':      0.1,
    'freq_weight':            0.05,

    'val_interval':           10,
    'val_set':                'Set14',
    'num_workers':            4,
    'resume_checkpoint':      RESUME_CKPT,
}

def get_patch_size(epoch: int, schedule: list) -> int:
    """Return patch size for given epoch based on schedule."""
    ps = schedule[0][1]
    for start_ep, size in schedule:
        if epoch >= start_ep:
            ps = size
    return ps

print('--- Round 2 Config ---')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Cell 6: Build Model, Data, Loss, Optimizer ─────────────────────────
import torch.optim as optim
import torch.nn as nn

raw_model = build_myelin_sr(CONFIG['scale'], CONFIG['quality']).to(device)
model = nn.DataParallel(raw_model) if torch.cuda.device_count() > 1 else raw_model
raw_model.print_architecture_summary()

# Progressive training: start with small patches (faster iteration)
initial_patch = get_patch_size(1, CONFIG['patch_schedule'])
train_loader = build_train_loader(
    str(DATA_DIR), CONFIG['scale'], initial_patch,
    CONFIG['batch_size'], CONFIG['num_workers'],
)
current_patch_size = initial_patch

val_loader = None
if (DATA_DIR / CONFIG['val_set']).exists():
    val_loader = build_val_loader(str(DATA_DIR), CONFIG['val_set'], CONFIG['scale'])
    print(f'Validation: {CONFIG["val_set"]}/image_SRF_{CONFIG["scale"]} ready')

# UPDATED: Charbonnier + FFT + VGG
criterion = MyelinSRLoss(
    perceptual_weight=CONFIG['perceptual_weight'],
    freq_weight=CONFIG['freq_weight'],
    use_perceptual=True,
    use_freq=True,
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=CONFIG['base_lr'], weight_decay=1e-4)

# FIX 2: ReduceLROnPlateau (no conflict with HomeostaticLR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=20, min_lr=1e-6
)

# NEW: Growth Manager
growth_manager = NetworkGrowthManager(
    model,
    growth_threshold=CONFIG['growth_threshold'],
    check_interval=CONFIG['growth_check_interval'],
    verbose=True,
)

start_epoch = 1
best_psnr   = 0.0
if CONFIG['resume_checkpoint']:
    ckpt = torch.load(CONFIG['resume_checkpoint'], map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state'], strict=False)
    best_psnr   = ckpt.get('best_psnr', 0.0)
    start_epoch = ckpt.get('epoch', 0) + 1
    print(f'Resumed from epoch {start_epoch-1}, best_psnr={best_psnr:.2f} dB')
    print('Optimizer NOT restored -- fresh momentum to escape plateau')

In [ ]:
# ── Cell 7: Homeostatic LR (cooperative mode) ──────────────────────────
import math

class HomeostaticLR:
    """
    Gradient-energy-adaptive per-batch LR.
    Cooperative with ReduceLROnPlateau: clamps to plateau_lr upper bound.
    """
    def __init__(self, lr_min=1e-6, alpha=0.05):
        self.lr_min  = lr_min
        self.alpha   = alpha
        self.ema_energy = 1.0

    def step(self, energy, plateau_lr):
        self.ema_energy = (1 - self.alpha) * self.ema_energy + self.alpha * max(energy, 1e-8)
        ratio  = energy / max(self.ema_energy, 1e-8)
        raw_lr = plateau_lr * math.pow(ratio, 0.5)
        return max(self.lr_min, min(plateau_lr, raw_lr))

homeostatic = HomeostaticLR(lr_min=1e-6)
print('HomeostaticLR ready (cooperative mode)')

In [ ]:
import json, time

log = []

for epoch in range(start_epoch, CONFIG['epochs'] + 1):
    # Helper to handle DataParallel wrapper for custom methods
    m_obj = model.module if isinstance(model, nn.DataParallel) else model

    # ── Progressive patch size (curriculum) ────────────────────────────
    new_patch = get_patch_size(epoch, CONFIG['patch_schedule'])
    if new_patch != current_patch_size:
        print(f'  [Curriculum] Epoch {epoch}: patch {current_patch_size}px -> {new_patch}px')
        train_loader = build_train_loader(
            str(DATA_DIR), CONFIG['scale'], new_patch,
            CONFIG['batch_size'], CONFIG['num_workers'],
        )
        current_patch_size = new_patch

    # ── Training step ──────────────────────────────────────────────────
    model.train()
    current_base_lr = optimizer.param_groups[0]['lr']
    ep_loss, n_batches = 0.0, 0
    t0 = time.time()

    for lr_imgs, hr_imgs in train_loader:
        lr_imgs, hr_imgs = lr_imgs.to(device), hr_imgs.to(device)

        sr_imgs = model(lr_imgs)
        loss, ld = criterion(sr_imgs, hr_imgs)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # Homeostatic LR
        plateau_lr = current_base_lr
        grad_energy = sum(
            p.grad.abs().mean().item() for p in model.parameters() if p.grad is not None
        ) / max(sum(1 for p in model.parameters() if p.grad is not None), 1)
        
        for pg in optimizer.param_groups:
            pg['lr'] = homeostatic.step(grad_energy, plateau_lr)

        optimizer.step()
        for pg in optimizer.param_groups:
            pg['lr'] = current_base_lr
        ep_loss += ld['total']
        n_batches += 1

    elapsed  = time.time() - t0
    avg_loss = ep_loss / max(n_batches, 1)
    entry    = {'epoch': epoch, 'loss': avg_loss, 'time_s': elapsed,
                'lr': optimizer.param_groups[0]['lr'], 'patch': current_patch_size}

    # ── Validation ─────────────────────────────────────────────────────
    val_psnr = 0.0
    if val_loader and epoch % CONFIG['val_interval'] == 0:
        model.eval()
        psnr_sum, ssim_sum, nv = 0.0, 0.0, 0
        with torch.no_grad():
            for lr_v, hr_v, _ in val_loader:
                sr_v = model(lr_v.to(device))
                psnr_sum += calculate_psnr(sr_v, hr_v.to(device), crop_border=CONFIG['scale'])
                ssim_sum += calculate_ssim(sr_v, hr_v.to(device), crop_border=CONFIG['scale'])
                nv += 1
        val_psnr = psnr_sum / max(nv, 1)
        val_ssim = ssim_sum / max(nv, 1)
        entry.update({'val_psnr': val_psnr, 'val_ssim': val_ssim})
        scheduler.step(val_psnr)

        print(f'Epoch {epoch:03d} | Loss={avg_loss:.4f} | PSNR={val_psnr:.2f}dB | '
              f'SSIM={val_ssim:.4f} | LR={optimizer.param_groups[0]["lr"]:.2e} | '
              f'patch={current_patch_size}px | {elapsed:.0f}s')

        if val_psnr > best_psnr:
            best_psnr = val_psnr
            # Save the raw model state (without .module prefix) for easier deployment
            torch.save({
                'epoch': epoch, 'model_state': m_obj.state_dict(),
                'optimizer_state': optimizer.state_dict(), 'best_psnr': best_psnr,
                'config': {'scale': CONFIG['scale'], 'quality': CONFIG['quality']},
            }, CKPT_DIR / 'myelin_sr_best.pt')
            print(f'  >> New best: {best_psnr:.2f} dB')
    else:
        if epoch % 10 == 0:
            print(f'Epoch {epoch:03d} | Loss={avg_loss:.4f} | LR={optimizer.param_groups[0]["lr"]:.2e} | {elapsed:.0f}s')

    # ── Sleep Consolidation ────────────────────────────────────────────
    if epoch % CONFIG['consolidation_interval'] == 0:
        m_obj.consolidate_all(rate=CONFIG['consolidation_rate'])
        sp = m_obj.get_total_sparsity() * 100
        stiff_vals = [m.get_avg_stiffness_fraction() for m in m_obj.modules() if isinstance(m, FPSANConv2d)]
        avg_stiff  = sum(stiff_vals) / max(len(stiff_vals), 1) * 100
        print(f'  [Sleep] Epoch {epoch}: sparsity={sp:.1f}% avg_stiffness={avg_stiff:.1f}%')

    # ── Neural Growth Check ────────────────────────────────────────────
    n_spawned = growth_manager.step(epoch)
    if n_spawned > 0:
        # Re-fetch parameters from the inner model
        optimizer.add_param_group({'params': [
            p for name, p in m_obj.named_parameters()
            if 'growth_child' in name and p not in {grp['params'][0] for grp in optimizer.param_groups}
        ], 'lr': optimizer.param_groups[0]['lr'] * 0.5})
        print(f'  [Growth] {n_spawned} new children added to optimizer')

    # ── Periodic Checkpoint ────────────────────────────────────────────
    if epoch % 50 == 0:
        torch.save({
            'epoch': epoch, 'model_state': m_obj.state_dict(),
            'optimizer_state': optimizer.state_dict(), 'best_psnr': best_psnr,
        }, CKPT_DIR / f'myelin_sr_epoch_{epoch}.pt')

    log.append(entry)

with open(CKPT_DIR / 'training_log_r2.json', 'w') as f:
    json.dump(log, f, indent=2)
print(f'\nTraining complete. Best PSNR: {best_psnr:.2f} dB')

In [ ]:
# ── Cell 9: Phase Gates + Growth Report ────────────────────────────────
validator = PhaseGateValidator()
validator.check_valid_output(model, device)
validator.check_sparsity(model)
if best_psnr >= 30.3:  validator.check_psnr_gate(best_psnr, 'beats_bicubic')
if best_psnr >= 30.5:  validator.check_psnr_gate(best_psnr, 'beats_lanczos')
if best_psnr >= 33.0:  validator.check_psnr_gate(best_psnr, 'matches_edsr')
validator.print_report()

# Growth report
stats = growth_manager.report()
print(f'\nGrowth summary:')
print(f'  Total children spawned: {stats["total_spawned"]}')
saturated = sum(1 for v in stats['layers'].values() if v['saturated'])
max_depth  = max((v['growth_depth'] for v in stats['layers'].values()), default=0)
print(f'  Saturated layers: {saturated}')
print(f'  Max growth tree depth: {max_depth}')

# Stiffness health
stiff_vals = [m.stiffness.mean().item() for m in model.modules() if isinstance(m, FPSANConv2d)]
avg_stiff  = sum(stiff_vals) / max(len(stiff_vals), 1)
print(f'\nAvg stiffness (raw): {avg_stiff:.4f} / {model.modules().__class__}')
if avg_stiff >= 5.5:
    print('[WARN] Stiffness still near max -- reduce consolidation_rate further')
else:
    print('[OK] Stiffness healthy')

In [ ]:
# ── Cell 10: Export to Ternary Binary ──────────────────────────────────
from export_model import export_ternary_binary

best_ckpt = CKPT_DIR / 'myelin_sr_best.pt'
if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state'], strict=False)
    model.eval().cpu()
    cfg = ckpt.get('config', {'scale': CONFIG['scale'], 'quality': CONFIG['quality']})

    bin_path = CKPT_DIR / 'myelin_sr_r2_deploy.bin'
    size = export_ternary_binary(model, bin_path, cfg)
    fp32_size = sum(p.numel() * 4 for p in model.parameters())

    print(f'Export complete:')
    print(f'  FP32 size:      {fp32_size/1024:.1f} KB')
    print(f'  Ternary binary: {size/1024:.1f} KB  ({fp32_size/size:.1f}x compression)')
    print(f'  Best PSNR:      {best_psnr:.2f} dB')
    print(f'  Saved:          {bin_path}')
else:
    print('[WARN] No best checkpoint found')